# Entraînement YOLO sur le dataset unified (68 classes) – Kaggle
Avant de lancer : **Settings → Accelerator → GPU T4 x2** et **Internet → On**.
**Add Input** : le dataset `unified_detection.zip` et le modèle `best.pt` (sert de point de départ).
Ensuite : **Run All**.

In [ ]:
!pip install -q -U ultralytics
import torch; print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN -> active le GPU dans Settings')

## 1. Trouver le dataset et best.pt dans /kaggle/input

In [ ]:
import glob, os, zipfile, shutil
# Kaggle dézippe souvent tout seul ; sinon on dézippe nous-mêmes
for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):
    print('Dézippe', z); zipfile.ZipFile(z).extractall('/kaggle/working/data')

yamls = glob.glob('/kaggle/input/**/data.yaml', recursive=True) + glob.glob('/kaggle/working/data/**/data.yaml', recursive=True)
weights = glob.glob('/kaggle/input/**/best.pt', recursive=True)
print('data.yaml :', yamls); print('best.pt :', weights)
assert yamls, 'data.yaml introuvable : as-tu ajouté le dataset avec Add Input ?'
DATA_DIR = os.path.dirname(yamls[0])
START = weights[0] if weights else 'yolo11s.pt'
print('Départ depuis :', START)

## 2. Réécrire data.yaml (les chemins pointaient vers /content de Colab)

In [ ]:
import yaml
cfg = yaml.safe_load(open(yamls[0]))
cfg['path'] = DATA_DIR
for split in ('train', 'val', 'test'):
    if split in cfg and isinstance(cfg[split], str):
        p = cfg[split]
        for prefix in ('/content/fusion_det/', '/content/'):
            p = p.replace(prefix, '')
        cfg[split] = p
yaml.safe_dump(cfg, open('/kaggle/working/data.yaml', 'w'))
print(open('/kaggle/working/data.yaml').read())

## 3. Entraîner (part de best.pt, nouvelles classes : 68)
60 epochs, ~2 h 30 sur T4 x2. Arrêt automatique si ça ne progresse plus.

In [ ]:
from ultralytics import YOLO
import torch
model = YOLO(START)
model.train(
    data='/kaggle/working/data.yaml', epochs=60, imgsz=640, batch=16, patience=15,
    device=0 if torch.cuda.device_count() == 1 else [0, 1],
    cos_lr=True, close_mosaic=10,
    scale=0.7, degrees=15, hsv_s=0.8, hsv_v=0.5, mixup=0.1, copy_paste=0.1,
    project='/kaggle/working/runs', name='unified1', exist_ok=True)

## 4. Score et export

In [ ]:
best = '/kaggle/working/runs/unified1/weights/best.pt'
m = YOLO(best).val(data='/kaggle/working/data.yaml')
print(f'mAP50 = {m.box.map50:.3f}   mAP50-95 = {m.box.map:.3f}')
shutil.copy(best, '/kaggle/working/best_unified1.pt')
print('Télécharge best_unified1.pt dans Output (panneau de droite) ou après Save Version.')